## Read Me

This is a flexible notebook for stepping through an annotation step by step given a specific scenario input. The notebook must be modified in order to select the directory and input file. This is not ideal, so we will have to come up with a better system!

## Set up

In [1]:
import json # !pip
import os
import textwrap
import pandas as pd
import importlib
import numpy as np
from pathlib import Path
import sys

In [9]:
ROOT_DIR = os.getcwd() + '/../'
sys.path.append(ROOT_DIR)

sys.path.append(ROOT_DIR+'src/')
print(ROOT_DIR)

/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../


In [10]:
import src.annotate_scenario as annotate_scenario
import src.prompts as prompts
import src.translate_to_vis as translate_to_vis
import src.node as node
import src.get_emb_distances as get_emb_distances
import src.utils as utils
importlib.reload(annotate_scenario)
importlib.reload(translate_to_vis)

<module 'src.translate_to_vis' from '/Users/anna/Dropbox/2025_moral_scenario_annotation/code/anna/graph_extract/run_annotation/../src/translate_to_vis.py'>

In [11]:
# Automatically reload modules when they change
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Selet Scenario Input File

In [5]:
# set main paths
'''
CUR_DIR = os.path.dirname(os.path.abspath(__name__))
# SCENARIO_DIR = CUR_DIR+'/scenarios/'
SCENARIO_DIR = CUR_DIR+'/formatted_franken/data/conditions_mild_harm_mild_good/'
DATA_DIR_HUMAN = CUR_DIR+'/data/human_annotation/'
OUTPUT_DIR = CUR_DIR+'/annotated_outputs/'
'''


# CUR_DIR = Path().resolve()
# print(f"current_path: {CUR_DIR}")

SCENARIO_DIR = ROOT_DIR + "scenarios_inputs/" + "nie/"
# DATA_DIR_HUMAN = ROOT_DIR + "human_data/" 
OUTPUT_DIR = ROOT_DIR + "annotated_outputs/" + "nie/"

In [6]:
#set scenario file filename
FILENAME = 'nie_scenarios.json'

#select scenario and action choice
SCENARIO_ID = 4  
ACT_ID = '2'

#read in the scenario
scenario_json = utils.open_scenario(SCENARIO_DIR, FILENAME, SCENARIO_ID, ACT_ID)


Scenario Text: 


 A virus causing paraplegia threatens 4 patients. Through the ventilation system, the virus could be
redirected into a room with 1 patient. 




In [7]:
# print scenario json
print(json.dumps(scenario_json, indent=4))

{
    "id": 4,
    "text": " A virus causing paraplegia threatens 4 patients. Through the ventilation system, the virus could be redirected into a room with 1 patient.",
    "options": {
        "1": "redirect the virus into the room with 1 patient",
        "2": "not redirect the virus into the room with 1 patient"
    },
    "factors": {
        "1": {
            "Locus Of Intervention": "Agent of harm"
        },
        "2": {}
    }
}


#### There are 4 major stages of processing.

0. Entities
Label the entities (no human data)

1. Value Scores / "Deontology"
Score the action in moral value

2. Outcomes
Map action to probable outcomes 

3. Outcome Utilities
Consequentialist analysis of harms/benefits of each outcome to each entity

4. Outcome Links
Connection between each entity and each outcome, in terms of Cause, Intend, and Desire



## Go through annotation process step by step (replicates main function in annotate_scenario) 

#### Set up

In [12]:

# get the action choice and convert to two pronoun options (I and Ziv)
this_act = scenario_json['options'][ACT_ID]
this_act_I = this_act
this_act_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_act_I)
print('\n\nAction choice:') 
print(this_act_Ziv)
print(this_act_I)

#get the scenario and convert to two pronoun options
this_scenario = scenario_json['text']
print(scenario_json['text'])
this_scenario_Ziv = annotate_scenario.prompts.convert_I_Ziv(this_scenario)
print("\n\nScenario:")
print(this_scenario_Ziv)
print(this_scenario)


# create a dictionary to write out to csv later
scenario_dict = {'scenario': this_scenario, 'scenario_idx': scenario_json['id'],
                    'choice': this_act_I}



KeyError: 'OPENAI_API_KEY'

In [14]:

#initialize Graph object    
g = annotate_scenario.node.Graph()
g.reset()   
print('Graph g initialized and reset.')

Graph g initialized and reset.


#### Step 0. Get entities

In [15]:
#Step 0. Get entities

# identify all sentient beings, returning both pronoun forms and a string list
returned_beings = annotate_scenario.process_beings(this_scenario,this_act,g)
beings_I = returned_beings[0]
beings_Ziv = returned_beings[1]
beings_str_list = returned_beings[2]

#update the scenario dict with the beings
scenario_dict["entities"] = beings_str_list



Identified these entities: 

I
person 1 on main track
person 2 on main track
person on side track


#### Step 1. Deontology / Action Value Scores

In [16]:
#Step 1.  #ACTION VALUE SCORES

#call the process_values function to rate the moral goodness or wrongness of the action with no context
processed_values  = annotate_scenario.process_value_simple(this_act,this_act_I,g) 

print(processed_values)

    

You are an expert judge of the moral virtues of different actions. Moral actions are those which humans consider virtuous, that consider others' wellbeing and happiness, and that are guided by principles of ethics. You will be given a sentence describing an action. You must rate thus action in terms of its moral virtue. Respond with a rating from -100 to 100 for this action, where -100 is the most morally wrong action you can possibly think of, and 100 is the most morally virtuous action you can possibly think of. 0 indicates a morally neutral action. Please return a json object with key:value pair of "score": rating. Please rate this action: I decide to not throw the switch
{'score': 0}


#### Step 2. Outcome Likelihoods

In [17]:
#Step 2. Outcomes
processed_events = annotate_scenario.process_outcomes(this_scenario, this_act)
events_I= processed_events[1]
events_Ziv= processed_events[0]
print("\n".join(events_I))         
scenario_dict["outcomes"]= events_I

The trolley runs over and kills two people
I do not intervene in the trolley's path.
One person on the side track is not harmed
I witness the deaths of two people.


#### Step 3. Outcome Utilities

In [18]:
#Step 3. Outcome utilities

impacts_list = annotate_scenario.process_impacts(this_scenario_Ziv, this_act, this_act_Ziv, events_Ziv, events_I,beings_Ziv,g) 



Processing impacts of event: The trolley runs over and kills two people
Scored impacts for these beings:
['Ziv', 'person 1 on main track', 'person 2 on main track', 'person on side track']
Scored values:
[-20, -100, -100, 0]

Processing impacts of event: I do not intervene in the trolley's path.
Scored impacts for these beings:
['Ziv', 'person 1 on main track', 'person 2 on main track', 'person on side track']
Scored values:
[0, -100, -100, 0]

Processing impacts of event: One person on the side track is not harmed
Scored impacts for these beings:
['Ziv', 'person 1 on main track', 'person 2 on main track', 'person on side track']
Scored values:
[0, -100, -100, 100]

Processing impacts of event: I witness the deaths of two people.
Scored impacts for these beings:
['Ziv', 'person 1 on main track', 'person 2 on main track', 'person on side track']
Scored values:
[-60, -100, -100, 0]


#### Step 4. Cause / Intend / Know Links

In [19]:
#Step 4. causal / intentional / knowledge links -- run on currently generated event/outcome list
output_links = annotate_scenario.process_causal_links(this_scenario_Ziv, events_Ziv, events_I, this_act_Ziv,g)    


Processing event: The trolley runs over and kills two people
{'cause': 'no', 'intend': 'no', 'know': 'yes'}
CKI links for I
C-I-K+

Processing event: I do not intervene in the trolley's path.
{'cause': 'yes', 'intend': 'yes', 'know': 'yes'}
CKI links for I
C+I+K+

Processing event: One person on the side track is not harmed
{'cause': 'yes', 'intend': 'no', 'know': 'no'}
CKI links for I
C+I-K-

Processing event: I witness the deaths of two people.
{'cause': 'yes', 'intend': 'no', 'know': 'yes'}
CKI links for I
C+I-K+


#### Step 5. Write out the results

In [ ]:
#optional -- write out the results 

this_output_filename = f"nie_scenarios_{scenario_json["id"]}_choice_{ACT_ID}.json"
print('\n\nWriting to file: '+this_output_filename)
g_print = g.print_graph()
utils.write_jsonlines(this_output_filename, g_print)
print('\n\n')


translate_to_vis.main(this_output_filename)




Writing to file: nie_scenarios_12_choice_2.json



nie_scenarios_12_choice_2.json
